# Pose Control based on Lyapunov Estabilization theory

## Configuring the environment

In [65]:
import numpy as np
import math
import matplotlib.pyplot as plt
from coppeliasim_zmqremoteapi_client import RemoteAPIClient
import os

# Parameters of Turtlebot 3
wheel_radius = 0.033
robot_width = 0.287

# Initialize the Remote API
client = RemoteAPIClient()
sim = client.require('sim')

# Open the turtlebot3 scene 
# simulation_file = os.getcwd()+'/turtlebot3_pose_estabilization.ttt'
# sim.loadScene(simulation_file)

# Use the stepping mode
sim.setStepping(True)

# Configure the handles
Turtlebot3 = sim.getObjectHandle('/Turtlebot3')
leftMotor = sim.getObjectHandle('/Turtlebot3/left_motor')
rightMotor = sim.getObjectHandle('/Turtlebot3/right_motor')
Goal = sim.getObjectHandle('/ReferenceFrame')

# Error tolerance
min_error = 0.05

# Robot Limits - considering the kinematic model
phi_wheel_max = 3.5

In [66]:
def draw_robot(x, y, theta, ax, scale=1.0, **kwargs):
    """
    Desenha o contorno do robô no gráfico fornecido.
    
    Parâmetros:
    x, y  : Posição do centro do robô (m)
    theta : Orientação do robô (rad)
    ax    : Objeto Axes do matplotlib onde o robô será desenhado
    scale : Escala do desenho (para ajustar o tamanho visualmente)
    **kwargs: Argumentos opcionais de plotagem (ex: color='b', linestyle='--')
    """
    p = np.zeros((12, 3))
    
    p[:] = [
        [ 1,    1/7,  1/scale],
        [-3/7,  1,    1/scale],
        [-5/7,  6/7,  1/scale],
        [-5/7,  5/7,  1/scale],
        [-3/7,  2/7,  1/scale],
        [-3/7,  0,    1/scale],
        [-3/7, -2/7,  1/scale],
        [-5/7, -5/7,  1/scale],
        [-5/7, -6/7,  1/scale],
        [-3/7, -1,    1/scale],
        [ 1,   -1/7,  1/scale],
        [ 1,    1/7,  1/scale]
    ]
    
    p = scale * p
    
    r = np.array([
        [np.cos(theta),  np.sin(theta)],
        [-np.sin(theta), np.cos(theta)],
        [x,              y]
    ])
    
    p_transf = np.dot(p, r)
    
    X_plot = p_transf[:, 0]
    Y_plot = p_transf[:, 1]
    
    if 'color' not in kwargs and 'c' not in kwargs:
        kwargs['color'] = 'blue'
        
    ax.plot(X_plot, Y_plot, **kwargs)

# Normalize angle to the range [-pi,pi)
def normalize_angle(angle):
    return np.mod(angle+np.pi, 2*np.pi) - np.pi



## Simulation Loop - Aicardi

In [ ]:
# Initialize the simulation
sim.startSimulation()

# Control gains
gamma = 10.0
h = 0
k = 0

while True:

    # Capture the robot pose from simulation
    TBPos=sim.getObjectPosition(Turtlebot3,-1)
    TBXOri=sim.getObjectOrientation(Turtlebot3,-1)
    qTurtlebot = np.array([TBPos[0], TBPos[1], math.radians(TBXOri[2])])

    # Capture the goal pose
    Goal_Pos = sim.getObjectPosition(Goal,-1)
    Goal_Ori = sim.getObjectOrientation(Goal,-1)
    qGoal = np.array([Goal_Pos[0], Goal_Pos[1], math.radians(Goal_Ori[2])])

    # Global States
    dx, dy, dth = qGoal - qTurtlebot

    # Transform to Aicardi states
    e = math.sqrt(dx**2 + dy**2)
    alpha = normalize_angle(-np.arctan2(dy,dx) + qTurtlebot[2])
    theta_aicardi = normalize_angle(-qGoal[2] + np.arctan2(dy,dx))

    # Stopping condition
    if e < min_error:
        # Goal achieved
        print("Alvo alcançado!")
        sim.stopSimulation()
        break

    # Calculate linear and angular velocities from (6) and (9)
    # u = gamma * cos(alpha) * e
    v = gamma * math.cos(alpha) * e
    
    # omega = k*alpha + gamma * (cos(alpha)*sin(alpha)/alpha) * (alpha + h*theta)
    omega = k * alpha + gamma * ((math.cos(alpha) * math.sin(alpha))/alpha) * (alpha + h * theta_aicardi)

    #  DDMR Inverse Kinematics
    w_right = (v + (omega * robot_width / 2)) / wheel_radius
    w_left  = (v - (omega * robot_width / 2)) / wheel_radius

    # Saturation Limits
    w_right = max(min(w_right, phi_wheel_max), -phi_wheel_max)
    w_left = max(min(w_left, phi_wheel_max), -phi_wheel_max)

    # Send commands
    sim.setJointTargetVelocity(leftMotor, w_left)
    sim.setJointTargetVelocity(rightMotor, w_right)

    print(f"Erro: {e} | Alpha: {alpha} | Theta: {theta_aicardi}")
    print(f"v: {v}, omega: {omega}")
    # Step the simulation
    sim.step()

Erro: 2.652512205438457 | Alpha: -2.2943833170481027 | Theta: 2.2669677492672973
v: 17.56167705480697, omega: -4.9618424211179635
Erro: 2.6487733904393043 | Alpha: -2.2955653638260665 | Theta: 2.2681420322744277
v: 17.56037559520748, omega: -4.96328610887244
Erro: 2.6445044657132026 | Alpha: -2.297070647360626 | Theta: 2.269647945147934
v: 17.56185617884188, omega: -4.965084424510746
Erro: 2.64027931101962 | Alpha: -2.2985525592015734 | Theta: 2.2711305408877482
v: 17.56303129791481, omega: -4.966810860348073
Erro: 2.636095534418014 | Alpha: -2.300032117408767 | Theta: 2.272612785374074
v: 17.56430378281723, omega: -4.968491028402615
Erro: 2.6318658271599147 | Alpha: -2.301554011577016 | Theta: 2.2741347619639427
v: 17.565968702729958, omega: -4.970173881472593
Erro: 2.6276891467612224 | Alpha: -2.3030598954318973 | Theta: 2.2756401894481195
v: 17.567538728053606, omega: -4.9717937081935535
Erro: 2.623520392887064 | Alpha: -2.304557756821813 | Theta: 2.27713947291503
v: 17.568872142834

KeyboardInterrupt: 